# Exercise 1 — Build your own SUT

In the previous notebooks we always *loaded* a table that already existed.
Here we do the opposite: we **build a Supply-Use Table (SUT) from scratch** to
describe a small real system — a single industrial plant — and then we let
MARIO compute its environmental footprint.

This is the classic **SUT Exercise 1**, reproduced step by step. The workflow is
always the same three moves:

1. **Describe** the system → list its *activities*, *commodities*, *units*…
2. **Fill in** the numbers → who supplies what, who uses what (in Excel).
3. **Parse & analyse** → load the filled file back into MARIO and read results.

> 💡 The real lesson is at the end: once the table is built, **changing a single
> input lets you re-run the entire exercise under a different assumption** —
> that is exactly how scenario analysis works.

## Step 0 — Import MARIO

Everything we need lives in the `mario` package.

In [1]:
import mario

## Step 1 — Describe your system

A SUT is organised by **sets** (its dimensions) and needs **units** of measure.
For our toy plant we define:

- **Commodities** — the products/energy carriers that flow in the system
  (fuels, heat, electricity, two physical products).
- **Activities** — the processes that *produce* those commodities
  (refinery, gas boiler, CHP plant, PV plant, industries).
- **Satellite account** — the environmental extension we care about: `CO2 emissions`.

The other sets (Factor of production, Consumption category, Region) are
**mandatory** but here we just give them a single sensible default.

`mario.write_parse_template(...)` then writes an **empty Excel template** with the
right rows and columns, ready to be filled by hand.

In [ ]:
# --- Define the SUT sets ----------------------------------------------------
sets = {
    # Commodities produced/consumed in the system
    "Commodity": [
        "fuel A",
        "fuel B",
        "heat",
        "electricity",
        "product A",
        "product B",
    ],
    # Activities (processes) that produce the commodities
    "Activity": [
        "refinery",
        "gas boiler",
        "chp plant",
        "pv plant",
        "industries",
    ],
    # Satellite account (environmental extension)
    "Satellite account": [
        "CO2 emissions",
    ],
    # --- Mandatory sets we leave at a simple default (editable) -------------
    "Factor of production": [
        "Value added",
    ],
    "Consumption category": [
        "Final demand",
    ],
    "Region": [
        "My plant",
    ],
}

# --- Units of measure (required for Commodity, Activity, Factor, Satellite) --
units = {
    # everything in MWh except product A and product B in tons
    "Commodity": {
        "fuel A": "MWh",
        "fuel B": "MWh",
        "heat": "MWh",
        "electricity": "MWh",
        "product A": "ton",
        "product B": "ton",
    },
    # main output unit of each activity
    "Activity": {
        "refinery": "MWh",
        "gas boiler": "MWh",
        "chp plant": "MWh",
        "pv plant": "MWh",
        "industries": "ton",
    },
    # value added in monetary units
    "Factor of production": "EUR",
    # CO2 emissions in tons
    "Satellite account": "ton",
}

# --- Write the empty Excel template to be filled in -------------------------
mario.write_parse_template(
    path="custom_sut_template.xlsx",
    table="SUT",
    sets=sets,
    units=units,
)

## Step 2 — Fill in the numbers (in Excel)

Open `custom_sut_template.xlsx`, fill the **Supply** block (who makes what),
the **Use** block (who consumes what), the value added and the CO2 emissions,
then save it as `custom_sut_template_filled.xlsx`.

> 💡 Nothing here is hidden in code: the data is just an Excel file you can edit
> with your eyes. This is what makes the next step so easy to repeat.

Now we read the filled file back into MARIO. The `tech_assumption` argument tells
MARIO how to interpret the table when it builds the derived matrices.

In [ ]:
my_plant = mario.parse_from_excel('custom_sut_template_filled.xlsx', 'SUT', 'flows', tech_assumption='product-based')

Notice the warning: we asked for the **product-based** assumption, but our table
is *not square* (5 activities vs 6 commodities), so MARIO automatically falls
back to the **industry-based** assumption. That is fine — and it is already a
preview of the big idea: the **technology assumption is just an input you can
change**.

## Step 3 — Inspect the database

The summary confirms MARIO read our system correctly: the right number of
activities, commodities, the satellite account, and a single region (`My plant`).

In [ ]:
my_plant

## Step 4 — Compute the CO2 footprint

This is the payoff. `my_plant.f` gives the **footprint intensities**: the total
(direct + indirect) CO2 emissions embodied in **one unit of output** of each
activity and each commodity.

MARIO resolves all the intermediate matrices on its own (you can see it in the
log) — we only asked for `f`.

In [ ]:
my_plant.f

## The key idea — change one input, redo everything

The whole exercise above is now a **machine you can re-run under new hypotheses**.
You don't rewrite the analysis — you change *one input* and execute the cells again:

| Change this input | …and you are testing |
|---|---|
| A number in the **filled Excel** (e.g. less gas, more PV) | a different **technology / energy mix** |
| The `tech_assumption` argument (`"product-based"` ↔ `"industry-based"`) | a different **construction assumption** |
| The **sets/units** in Step 1 | a different **system boundary** (add an activity, a commodity, a pollutant) |

Because the structure is fixed and the data lives in a file, every "what if?"
question becomes: *edit one thing → re-run → compare the new `f`.* That is the
essence of scenario analysis with SUTs.

### 🧩 Your turn

1. In the filled Excel, **halve the gas boiler output** and shift it to the PV
   plant. Re-run from Step 2 and look at how the CO2 footprint of `heat` and
   `electricity` changes.
2. Re-parse the same file with `tech_assumption='industry-based'` explicitly and
   confirm you get the same result as the automatic fallback.
